<a href="https://colab.research.google.com/github/2001lida/PythonLession2/blob/hw_6/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B56.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Titanic: бинарная классификация признака `Survived`

**Выбранная метрика:**  
**ROC-AUC** — основная метрика, потому что задача бинарной классификации имеет небольшой дисбаланс классов, а ROC-AUC оценивает качество ранжирования предсказаний независимо от порога.  
**Accuracy** используется как дополнительная интерпретируемая метрика, чтобы было проще сравнить модель с константным бейзлайном.


In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

RANDOM_STATE = 42

df = pd.read_csv("train.csv")
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:

print("Размер датасета:", df.shape)
print("\nПропуски:")
print(df.isna().sum().sort_values(ascending=False))

print("\nРаспределение целевого признака:")
print(df["Survived"].value_counts(normalize=True))


Размер датасета: (891, 12)

Пропуски:
Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

Распределение целевого признака:
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


## Подготовка признаков

Для модели исключим:
- `Survived` — целевой признак;
- `PassengerId` — технический идентификатор;
- `Name`, `Ticket`, `Cabin` — признаки, которые либо слишком текстовые, либо содержат много пропусков и требуют отдельного feature engineering.

Оставим базовые признаки:
- числовые: `Pclass`, `Age`, `SibSp`, `Parch`, `Fare`;
- категориальные: `Sex`, `Embarked`.


In [3]:

target_col = "Survived"

feature_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]

X = df[feature_cols].copy()
y = df[target_col].copy()

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (712, 7)
Test shape: (179, 7)


In [4]:

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]
)

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

# Для ROC-AUC нужны вероятности положительного класса
baseline_proba = baseline_model.predict_proba(X_test)[:, 1]

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_roc_auc = roc_auc_score(y_test, baseline_proba)

print("Baseline results")
print("Accuracy:", round(baseline_accuracy, 4))
print("ROC-AUC :", round(baseline_roc_auc, 4))


Baseline results
Accuracy: 0.6145
ROC-AUC : 0.5


## Обучение ML-модели

Используем `LogisticRegression`:
- это сильный и интерпретируемый baseline для бинарной классификации;
- модель хорошо работает с небольшими табличными данными;
- в пайплайне учтены пропуски и one-hot кодирование категориальных признаков.


In [5]:

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred)
roc_auc = roc_auc_score(y_test, proba)

print("LogisticRegression results")
print("Accuracy:", round(accuracy, 4))
print("ROC-AUC :", round(roc_auc, 4))


LogisticRegression results
Accuracy: 0.8045
ROC-AUC : 0.8437


In [6]:

print("Classification report:\n")
print(classification_report(y_test, pred))


Classification report:

              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

